[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amistein/ml-study-group/blob/main/notebooks/lesson_004.ipynb)

### What is Gradient Descent?

In lesson 2, we plotted our loss function and saw that it makes a nice U-shaped curve. We could look at the graph and see that the lowest point was at around 145 (price per sqft). But eyeballing a graph is a strategy for humans, not for computers. As we've been saying, we need a way to find that minimum automatically, and we're ready to do that now.

So what's the idea? Let's say our current weight is 50. We know from the graph that the minimum is somewhere to the right. But how would we know that without looking at the graph? Well, we could try nudging the weight up by a tiny amount, say from 50 to 50.001, and see what happens to the loss. If the loss goes *down*, we know we moved in the right direction. If it goes *up*, we know we should have gone the other way. But we can do better than just knowing the direction. We can also figure out *how steep* the curve is at our current position. If the curve is steep, we're far from the minimum and we should take a bigger step. If the curve is nearly flat, then we're close to the minimum and should take a smaller step. The steepness of the curve at a given point is called the **slope** or **gradient** at that point. (There technically is a difference between the slope and the gradient, which we'll get into at a later point, but for now we'll use them interchangably).  

In case you're not really familiar with the concept of the slope of a line, it's how much the line changes on the `y` axis relative to a change on the `x` axis. In the graph below, if we increase `x` by 1, `y` increases by 0.5. In other words, the slope is equal to the change in `y` divided by the change in `x`. Formally, the formula is known as "rise over run". **Rise** is the change on the `y` axis, and **run** in the change on the `x` axis. `slope = rise / run`.

<img src="../images/slope_line.png" width="500">

The graph above is showing a straight line, and so the slope is the same at every point. But if we plot the output of a non-linear function, like our U-shaped loss function (or the function `f(x) = x^2`), then the slope is not be the same at every point. And for gradient descent (i.e. figuring out how much to move our weight and in which direction), we need to know what the slope is at the point we are currently at. So if we take the value of our current weight and we increase it by 0.001 and the loss drops by 1000, that's a steep downward slope. If the loss only drops by 0.01, the slope is much gentler. We can compute this as a simple ratio:

```
slope = (loss_at_new_weight - loss_at_old_weight) / (new_weight - old_weight)
```

Or in shorter form, if we call the tiny nudge amount `h`:

```
slope = (f(x + h) - f(x)) / h
```

This is called a **numerical gradient**, and it works for any function. You don't need to know any calculus to compute it. You just need to be able to call the function twice. (In future lessons, we'll introduce the more technically precise gradient from Calculus, but the intuition is the same, so we're starting with a simpler version).

Once we have the slope, the update rule is simple: move the weight in the opposite direction of the slope (because if the slope is negative the loss is decreasing and we want to keep going, and if it's positive the loss is increasing and we want to go back). And one last step: we also scale the step by a number called the **learning rate**, which controls how big our steps are. This learning rate can be adjusted to take bigger or smaller steps as needed. So the final update looks like this:

```
new_weight = weight - learning_rate * slope
```

Every machine learning algorithm uses this basic approach to updating it's weights. In a loop, we keep adjusting the weigths in the opposite direction of the gradient, until our loss becomes low enough, or for a set amount of steps. Except that for most models, we have to do this for each of several or many weights, and not just one.

<img src="../images/pos_and_neg_loss.png" width="500">

### **Exercises**
#### Implement the following functions, and ensure the tests pass.

*Reading through the test cases is a good way to make sure you understand what your code is doing.*

First, let's set up our loss function from lesson 2 so we have something to work with.

In [ ]:
# --- Setup (do not modify) ---
sqft = [800, 1200, 1500, 2000, 2500]
actual_prices = [150000, 210000, 260000, 340000, 420000]

def mse(weight):
    """Mean squared error for our housing data, as a function of price_per_sqft."""
    predictions = [weight * s for s in sqft]
    return sum((predictions[i] - actual_prices[i]) ** 2 for i in range(len(actual_prices))) / len(actual_prices)

Now implement a function that computes the numerical gradient of *any* function at a given point. This is the slope formula from above: nudge `x` up by a tiny amount `h`, see how much `f` changes, and divide by `h`.

In [ ]:
def numerical_gradient(f, x, h=0.001):
    """Compute the slope of function f at point x.

    Args:
        f: a function that takes a number and returns a number
        x: the point at which to compute the slope
        h: the tiny nudge amount (default 0.001)
    Returns:
        the slope of f at x
    """
    # YOUR CODE HERE
    raise NotImplementedError()


# --- Tests (do not modify) ---
# Test with a simple function first: f(x) = x^2
# The slope of x^2 at x=3 should be about 6 (it's exactly 6.001 with h=0.001)
def simple_square(x):
    return x ** 2

_g1 = numerical_gradient(simple_square, 3)
assert abs(_g1 - 6.001) < 0.0001, f"Expected ~6.001, got {_g1}"

# Slope of x^2 at x=-2 should be about -4
_g2 = numerical_gradient(simple_square, -2)
assert abs(_g2 - (-3.999)) < 0.0001, f"Expected ~-3.999, got {_g2}"

# Slope of x^2 at x=0 should be about 0 (the minimum!)
_g3 = numerical_gradient(simple_square, 0)
assert abs(_g3) < 0.01, f"Expected ~0, got {_g3}"

# Now test with our MSE loss function
# At weight=50, the slope should be steeply negative (we need to go higher)
_g4 = numerical_gradient(mse, 50)
assert _g4 < -700000000, f"Expected a large negative gradient at 50, got {_g4}"

# At weight=200, the slope should be positive (we went too far)
_g5 = numerical_gradient(mse, 200)
assert _g5 > 100000000, f"Expected a large positive gradient at 200, got {_g5}"

print(f"Slope of x^2 at x=3:  {_g1:.3f} (should be ~6)")
print(f"Slope of x^2 at x=-2: {_g2:.3f} (should be ~-4)")
print(f"Slope of x^2 at x=0:  {_g3:.3f} (should be ~0)")
print(f"Slope of MSE at w=50:  {_g4:,.0f} (negative = go higher)")
print(f"Slope of MSE at w=200: {_g5:,.0f} (positive = go lower)")
print("✅ All tests passed!")

Now let's put it together. Implement gradient descent: start with an initial weight, and repeatedly nudge it in the direction that reduces the loss. The function should return the list of weights at each step so we can see how the weight evolves over time.

In [ ]:
def gradient_descent(f, initial_weight, learning_rate, num_steps):
    """Run gradient descent to minimize function f.

    Args:
        f: the loss function to minimize (takes a number, returns a number)
        initial_weight: the starting weight
        learning_rate: how big each step is
        num_steps: how many steps to take
    Returns:
        list of weights at each step (length = num_steps + 1, including the initial weight)
    """
    # YOUR CODE HERE
    # Hint: start with the initial weight, and at each step:
    # 1. Compute the gradient using numerical_gradient
    # 2. Update: weight = weight - learning_rate * gradient
    # 3. Store the new weight
    raise NotImplementedError()


# --- Tests (do not modify) ---
# Run gradient descent on our housing MSE
_weights = gradient_descent(mse, initial_weight=50, learning_rate=0.0000001, num_steps=20)

# Should return 21 weights (initial + 20 steps)
assert len(_weights) == 21, f"Expected 21 weights, got {len(_weights)}"

# First weight should be our initial weight
assert _weights[0] == 50, f"First weight should be 50, got {_weights[0]}"

# The weight should be increasing (moving toward ~171)
assert _weights[-1] > _weights[0], "Weight should be increasing toward the minimum"

# After 20 steps with this learning rate, we should be close to the optimal (~170.9)
assert abs(_weights[-1] - 170.9) < 1, f"Expected weight near 170.9, got {_weights[-1]:.4f}"

# Loss should have decreased
assert mse(_weights[-1]) < mse(_weights[0]), "Loss should decrease during training"

print(f"Starting weight: {_weights[0]}")
print(f"Final weight:    {_weights[-1]:.4f}")
print(f"Starting loss:   {mse(_weights[0]):,.0f}")
print(f"Final loss:      {mse(_weights[-1]):,.0f}")
print("✅ All tests passed!")

Let's visualize what just happened. Plot the loss at each step to see how gradient descent converges on the minimum.


In [ ]:
from matplotlib import pyplot as plt

# YOUR CODE HERE: compute the loss at each weight in _weights
losses = ...

# --- Plot (do not modify) ---
plt.plot(range(len(losses)), losses)
plt.xlabel("Step")
plt.ylabel("MSE Loss")
plt.title("Gradient Descent Convergence")
plt.show()

You should see the loss drop steeply at first, then flatten out as we approach the minimum. This shape is typical of gradient descent: the gradient is large when we're far from the minimum (so we take big steps), and small when we're close (so we take small steps). It naturally slows down as it converges.